# Logistic Regression – Wine Dataset

Uses the **Wine dataset** built into Python through `scikit-learn`, but the actual preprocessing and modeling use your custom package.

The package functionality used here is:

```python
from ml_package.preprocessing import train_test_split, standardize
from ml_package.supervised_learning.logistic_models import LogisticRegression
```

## Classification task

The Wine dataset contains 178 wine samples with 13 numeric chemistry features and 3 cultivar classes. We predict the cultivar class using all 13 features.

## Goals

By the end of this notebook, you should be able to:

1. Load and inspect the Wine dataset.
2. Use your package's `train_test_split`.
3. Use your package's `standardize`.
4. Fit your package's `LogisticRegression` in multinomial and one-vs-rest modes.
5. Evaluate classifiers using accuracy and confusion matrices.
6. Compare the effect of L2 regularization.
7. Interpret predicted probabilities.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine

# Ensure the package root is on the path
cwd = Path.cwd()
for root in [cwd, cwd.parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

from ml_package.processing.preprocessing import train_test_split, standardize
from ml_package.supervised.logistic_regression import LogisticRegression

## 2. Load the Wine Dataset

We use `sklearn.datasets.load_wine()` only to access the dataset.
The model itself will come from your custom package.

The dataset contains:

- 178 wine samples
- 13 numeric chemistry features
- 3 wine cultivar classes (0, 1, 2)

In [ ]:
wine = load_wine(as_frame=True)

wine_features = wine.data.copy()
wine_target = wine.target.copy()
class_names = np.array(wine.target_names)

X = wine_features.to_numpy()
y = wine_target.to_numpy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Classes:", class_names)
print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))

wine_features.head()

## 3. Exploratory Data Analysis

In [ ]:
wine_features.describe().T

In [ ]:
# Per-class feature means — helps identify which features separate classes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for class_id, ax in enumerate(axes):
    mask = y == class_id
    ax.bar(wine_features.columns, wine_features[mask].mean(), color=f"C{class_id}")
    ax.set_title(f"Class {class_id}: {class_names[class_id]}")
    ax.set_xticklabels(wine_features.columns, rotation=90, fontsize=7)
    ax.set_ylabel("Mean value")

plt.suptitle("Per-Class Feature Means", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot of the two most discriminating features
fig, ax = plt.subplots(figsize=(7, 5))

for class_id in range(3):
    mask = y == class_id
    ax.scatter(
        wine_features.loc[mask, "flavanoids"],
        wine_features.loc[mask, "proline"],
        label=class_names[class_id],
        alpha=0.7,
        color=f"C{class_id}",
    )

ax.set_xlabel("Flavanoids")
ax.set_ylabel("Proline")
ax.set_title("Flavanoids vs. Proline by Cultivar")
ax.legend()
plt.show()

## 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)

## 5. Standardize Features

Logistic regression is scale-sensitive. We standardize the training set and apply the same parameters to the test set to avoid data leakage.

In [ ]:
X_train_scaled, scaling_params = standardize(X_train, return_params=True)
X_test_scaled = standardize(
    X_test,
    mean=scaling_params["mean"],
    scale=scaling_params["scale"],
)

print("Training feature means after scaling:")
print(np.round(X_train_scaled.mean(axis=0), 6))

print("\nTraining feature standard deviations after scaling:")
print(np.round(X_train_scaled.std(axis=0), 6))

## 6. Evaluation Helper Functions

In [ ]:
def accuracy(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def confusion_matrix(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def plot_confusion_matrix(cm, class_names, title="Confusion Matrix"):
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.colorbar(im, ax=ax)
    ax.set(
        xticks=range(len(class_names)), yticks=range(len(class_names)),
        xticklabels=class_names, yticklabels=class_names,
        xlabel="Predicted", ylabel="Actual", title=title,
    )
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.tight_layout()
    plt.show()


def evaluate(model, X_train, y_train, X_test, y_test):
    return {
        "Train Accuracy": accuracy(y_train, model.predict(X_train)),
        "Test Accuracy":  accuracy(y_test,  model.predict(X_test)),
        "Iterations":     model.n_iter_,
    }

## 7. Multinomial Softmax Regression

Trains a single model with a softmax output layer. This is the `multi_class='multinomial'` mode.

The model minimizes:

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\sum_{k=1}^{K} y_{ik} \log \hat{p}_{ik}$$

In [ ]:
softmax_model = LogisticRegression(
    learning_rate=0.1,
    max_iter=1000,
    multi_class="multinomial",
    standardize=False,  # already standardized using preprocessing.standardize()
    random_state=42,
)

softmax_model.fit(X_train_scaled, y_train)

softmax_results = evaluate(softmax_model, X_train_scaled, y_train, X_test_scaled, y_test)
pd.DataFrame([softmax_results], index=["Multinomial"])

In [ ]:
cm = confusion_matrix(y_test, softmax_model.predict(X_test_scaled), n_classes=3)
plot_confusion_matrix(cm, class_names, title="Multinomial – Confusion Matrix")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(softmax_model.loss_history_)
plt.xlabel("Iteration")
plt.ylabel("Cross-Entropy Loss")
plt.title("Multinomial – Training Loss Curve")
plt.show()

## 8. One-vs-Rest Regression

Trains one binary sigmoid classifier per class. The class with the highest score wins.

This is the `multi_class='ovr'` mode.

In [ ]:
ovr_model = LogisticRegression(
    learning_rate=0.1,
    max_iter=1000,
    multi_class="ovr",
    standardize=False,
    random_state=42,
)

ovr_model.fit(X_train_scaled, y_train)

ovr_results = evaluate(ovr_model, X_train_scaled, y_train, X_test_scaled, y_test)
pd.DataFrame([ovr_results], index=["OvR"])

In [ ]:
cm_ovr = confusion_matrix(y_test, ovr_model.predict(X_test_scaled), n_classes=3)
plot_confusion_matrix(cm_ovr, class_names, title="One-vs-Rest – Confusion Matrix")

## 9. Effect of L2 Regularization

The `l2_penalty` parameter adds an L2 penalty to the loss:

$$\mathcal{L}_{\text{reg}} = \mathcal{L} + \frac{\lambda}{2} \sum_{j} \beta_j^2$$

Higher penalty shrinks coefficients and can reduce overfitting. The intercept is never penalized.

In [ ]:
penalties = [0.0, 0.01, 0.1, 1.0, 10.0]
reg_rows = []

for lam in penalties:
    model = LogisticRegression(
        learning_rate=0.1,
        max_iter=1000,
        multi_class="multinomial",
        standardize=False,
        l2_penalty=lam,
        random_state=42,
    )
    model.fit(X_train_scaled, y_train)
    reg_rows.append({
        "l2_penalty": lam,
        "Train Accuracy": accuracy(y_train, model.predict(X_train_scaled)),
        "Test Accuracy":  accuracy(y_test,  model.predict(X_test_scaled)),
        "Iterations": model.n_iter_,
    })

reg_df = pd.DataFrame(reg_rows)
reg_df

In [ ]:
plt.figure(figsize=(7, 4))
x_vals = [1e-4 if v == 0 else v for v in reg_df["l2_penalty"]]
plt.semilogx(x_vals, reg_df["Train Accuracy"], marker="o", label="Train")
plt.semilogx(x_vals, reg_df["Test Accuracy"],  marker="s", label="Test")
plt.xlabel("L2 Penalty (log scale, 0 shown as 1e-4)")
plt.ylabel("Accuracy")
plt.title("Accuracy vs. L2 Regularization Strength")
plt.legend()
plt.show()

## 10. Compare All Strategies

In [ ]:
models = {
    "Multinomial":          LogisticRegression(learning_rate=0.1, max_iter=1000, multi_class="multinomial", standardize=False, random_state=42),
    "OvR":                  LogisticRegression(learning_rate=0.1, max_iter=1000, multi_class="ovr",         standardize=False, random_state=42),
    "Multinomial + L2=0.1": LogisticRegression(learning_rate=0.1, max_iter=1000, multi_class="multinomial", standardize=False, l2_penalty=0.1, random_state=42),
    "OvR + L2=0.1":         LogisticRegression(learning_rate=0.1, max_iter=1000, multi_class="ovr",         standardize=False, l2_penalty=0.1, random_state=42),
}

comparison_rows = []
fitted = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    comparison_rows.append({"Model": name, **evaluate(model, X_train_scaled, y_train, X_test_scaled, y_test)})
    fitted[name] = model

comparison_df = pd.DataFrame(comparison_rows).sort_values("Test Accuracy", ascending=False).reset_index(drop=True)
comparison_df

In [ ]:
x = range(len(comparison_df))
plt.figure(figsize=(8, 4))
plt.bar([i - 0.2 for i in x], comparison_df["Train Accuracy"], width=0.4, label="Train")
plt.bar([i + 0.2 for i in x], comparison_df["Test Accuracy"],  width=0.4, label="Test")
plt.xticks(list(x), comparison_df["Model"], rotation=15, ha="right")
plt.ylabel("Accuracy")
plt.title("Model Comparison")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Interpret Coefficients

Because the predictors were standardized, each coefficient can be interpreted as:

> the expected change in log-odds for a one-standard-deviation increase in that feature, holding all others fixed.

We use the best model from the comparison above.

In [ ]:
best_name = comparison_df.iloc[0]["Model"]
best_model = fitted[best_name]
print("Best model:", best_name)

coef_df = pd.DataFrame(
    best_model.coef_,
    index=[f"Class {i}: {class_names[i]}" for i in range(3)],
    columns=wine_features.columns,
)
coef_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for class_id, ax in enumerate(axes):
    coefs = best_model.coef_[class_id]
    sorted_idx = np.argsort(coefs)
    ax.barh(wine_features.columns[sorted_idx], coefs[sorted_idx], color=f"C{class_id}")
    ax.axvline(0, linewidth=1)
    ax.set_title(f"Class {class_id}: {class_names[class_id]}")
    ax.set_xlabel("Coefficient")

plt.suptitle(f"Coefficients – {best_name}")
plt.tight_layout()
plt.show()

## 12. Predicted Probabilities

Each row in `predict_proba()` sums to 1. The predicted class is the one with the highest probability.

In [ ]:
proba = best_model.predict_proba(X_test_scaled)
proba_df = pd.DataFrame(proba, columns=[f"P(class {i})" for i in range(3)])
proba_df["Predicted"] = best_model.predict(X_test_scaled)
proba_df["Actual"]    = y_test
proba_df["Correct"]   = proba_df["Predicted"] == proba_df["Actual"]
proba_df.head(10)

In [ ]:
correct_conf   = proba[proba_df["Correct"]].max(axis=1)
incorrect_conf = proba[~proba_df["Correct"]].max(axis=1)

plt.figure(figsize=(7, 4))
plt.hist(correct_conf,   bins=15, alpha=0.6, label=f"Correct (n={len(correct_conf)})")
plt.hist(incorrect_conf, bins=15, alpha=0.6, label=f"Incorrect (n={len(incorrect_conf)})")
plt.xlabel("Max Predicted Probability")
plt.ylabel("Count")
plt.title("Prediction Confidence: Correct vs. Incorrect")
plt.legend()
plt.show()

## 13. Prediction Example on One Wine

In [ ]:
example_index = 0
x_example  = X_test_scaled[example_index].reshape(1, -1)
true_label  = y_test[example_index]
pred_label  = best_model.predict(x_example)[0]
pred_proba  = best_model.predict_proba(x_example)[0]

print(f"True class:      {true_label} ({class_names[true_label]})")
print(f"Predicted class: {pred_label} ({class_names[pred_label]})")
print(f"Correct:         {true_label == pred_label}")
print()
for i, p in enumerate(pred_proba):
    print(f"  P(class {i} – {class_names[i]}): {p:.4f}")

## 14. Summary

In this notebook, we used your custom machine learning package to complete a logistic regression workflow.

### Package functions/classes used

From preprocessing:

```python
train_test_split
standardize
```

From logistic models:

```python
LogisticRegression  # multi_class="multinomial" | "ovr"
```

### Main workflow

1. Loaded the Wine dataset.
2. Predicted cultivar class (0, 1, 2) from 13 chemistry features.
3. Split the data using your package.
4. Standardized features using your package.
5. Fit multinomial softmax and one-vs-rest classifiers using your package.
6. Compared the effect of L2 regularization strength.
7. Evaluated models with accuracy and confusion matrices.
8. Interpreted standardized coefficients per class.

### Interpretation note

Because the predictors were standardized, coefficient size reflects the effect of a one-standard-deviation change in that predictor on the log-odds of that class. Larger absolute values indicate more influential features.